# Claude Agent SDK: using Claude Code features (CLAUDE.md, skills, hooks, settings)

The Agent SDK is built on the same foundation as Claude Code, so your SDK agents can use the **same filesystem-based features**: project instructions (`CLAUDE.md` and rules), skills, hooks, and settings.

This notebook makes those features **visible**. You will run the *same question* with and without them and watch the behavior change.

**What you will learn (and see running):**
1. `setting_sources`: the switch that decides which files the agent reads
2. `CLAUDE.md`: persistent project instructions
3. Skills: knowledge and workflows loaded **on demand**
4. Hooks: programmatic callbacks vs `settings.json` shell hooks
5. What `setting_sources` does *not* control (important for multi-tenant apps)
6. Which feature to use for which goal

## The mental model

```
   ClaudeAgentOptions(setting_sources=[...])
                │
                ▼
   ┌─────────────── what gets loaded into the agent ───────────────┐
   │  "user"    ~/.claude/        CLAUDE.md, rules, skills, settings │
   │  "project" <cwd>/.claude/    CLAUDE.md, rules, skills, hooks    │
   │  "local"   <cwd>/.claude/    CLAUDE.local.md, settings.local    │
   └────────────────────────────────────────────────────────────────┘
```

- **Omit** `setting_sources` and the SDK behaves like the Claude Code CLI: it loads `user`, `project` **and** `local`.
- Pass `setting_sources=[]` to run with **only what you configure in code**.
- Pass an explicit list (for example `["project"]`) to opt in to specific sources.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("cc-features-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Build a project that has instructions and a skill

We create three files inside the demo folder:

| File | Feature |
|---|---|
| `CLAUDE.md` | Project instruction: *always start replies with the tag `[HOUSE-RULES]`* |
| `.claude/skills/flurbo-glossary/SKILL.md` | A skill that defines an invented term ("flurbo") that Claude cannot possibly know |
| `.claude/settings.json` | (Later) a shell hook |

Using an invented term and an unusual tag makes it unmistakable when the feature is active.

In [ ]:
CLAUDE_MD = """# Project rules

- Begin every reply with the exact tag [HOUSE-RULES].
- Keep answers to two sentences or fewer.
"""

SKILL_MD = """---
name: flurbo-glossary
description: Use when the user asks what a "flurbo" is, or about Acme build terminology.
---

# Flurbo glossary

At Acme, a **flurbo** is a nightly build that failed and is automatically
retried up to 3 times before an engineer is paged.
"""

def build_project():
    (DEMO_DIR / "CLAUDE.md").write_text(CLAUDE_MD)
    skill_dir = DEMO_DIR / ".claude" / "skills" / "flurbo-glossary"
    skill_dir.mkdir(parents=True, exist_ok=True)
    (skill_dir / "SKILL.md").write_text(SKILL_MD)

build_project()

for p in sorted(DEMO_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(DEMO_DIR))

A **skill** is just a folder with a `SKILL.md`: YAML front matter (`name`, `description`) plus instructions. The agent only sees the **short description** at startup and loads the full text **when it decides the skill is relevant**, which keeps context small. (`CLAUDE.md`, by contrast, loads in full every session.)

> Skills are filesystem artifacts. The SDK has **no programmatic API to register a skill**.

In [ ]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, ResultMessage, ToolUseBlock,
)

async def ask(prompt, **opts):
    """Run one query in the demo folder; print the tools used and the final answer."""
    opts.setdefault("cwd", str(DEMO_DIR))
    answer, tools = "", []
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                tools += [b.name for b in m.content if isinstance(b, ToolUseBlock)]
            elif isinstance(m, ResultMessage) and m.subtype == "success":
                answer = m.result
    except Exception as error:
        print("(query raised:", str(error)[:150], ")")
    print("tools used:", tools)
    print("answer    :", answer)
    return answer, tools

QUESTION = "What is a flurbo at Acme?"

## 2. Experiment A: `setting_sources=[]` (nothing loaded)

The agent sees **no** `CLAUDE.md` and **no** skills. Expect: it does not know what a flurbo is, and the `[HOUSE-RULES]` tag is **absent**.

In [ ]:
answer_a, _ = await ask(QUESTION, setting_sources=[], allowed_tools=["Read", "Glob"])

print()
print("tag present   :", "[HOUSE-RULES]" in answer_a)
print("knows flurbo  :", "retr" in answer_a.lower())

## 3. Experiment B: `setting_sources=["project"]` plus skills

Now the SDK loads `<cwd>/CLAUDE.md` **and** discovers `.claude/skills/`. Expect: the tag appears (instruction followed) and the answer matches the skill (the agent invoked the `Skill` tool).

`skills="all"` enables every discovered skill. (When you set `skills`, the SDK adds the `Skill` tool to `allowed_tools` for you.)

In [ ]:
answer_b, tools_b = await ask(
    QUESTION,
    setting_sources=["project"],
    skills="all",
    allowed_tools=["Read", "Glob"],
)

print()
print("tag present   :", "[HOUSE-RULES]" in answer_b)
print("knows flurbo  :", "retr" in answer_b.lower())
print("used Skill    :", "Skill" in tools_b)

Same question, same model, same code. The only difference was **which files the SDK loaded**. That is the whole idea of `setting_sources`.

## 4. Experiment C: CLAUDE.md without skills

`skills=[]` disables all skills, while `setting_sources=["project"]` still loads `CLAUDE.md`. Expect: the tag **is** present, but the agent still cannot define "flurbo".

In [ ]:
answer_c, _ = await ask(
    QUESTION,
    setting_sources=["project"],
    skills=[],
    allowed_tools=["Read", "Glob"],
)

print()
print("tag present   :", "[HOUSE-RULES]" in answer_c)
print("knows flurbo  :", "retr" in answer_c.lower())

| Run | `CLAUDE.md` | Skill | Tag shown | Knows "flurbo" |
|---|:-:|:-:|:-:|:-:|
| A: `setting_sources=[]` | no | no | no | no |
| B: `["project"]` + `skills="all"` | yes | yes | yes | yes |
| C: `["project"]` + `skills=[]` | yes | no | yes | no |

## 5. The three sources in detail

| Source | What it loads | Where |
|---|---|---|
| `"project"` | Project `settings.json` and hooks; project `CLAUDE.md` and `.claude/rules/*.md`; project skills, commands, subagents | `<cwd>/.claude/` for settings and hooks; `<cwd>` **and every parent directory** for `CLAUDE.md` and rules; `<cwd>` and parents up to the repo root for skills, commands, subagents |
| `"user"` | User settings; user `CLAUDE.md` and rules; user skills, commands, subagents | `~/.claude/` |
| `"local"` | `CLAUDE.local.md` and `.claude/settings.local.json` (personal, usually git-ignored) | `<cwd>/.claude/` and `<cwd>` plus parents |

The **`cwd` option decides where project-level inputs are found**. Project `settings.json` and hooks load only from `<cwd>/.claude/` (no parent fallback).

### `CLAUDE.md` load locations

| Level | Location | Loaded when |
|---|---|---|
| Project root | `<cwd>/CLAUDE.md` or `<cwd>/.claude/CLAUDE.md` | `"project"` included |
| Project rules | `<cwd>/.claude/rules/*.md` (and in parent dirs) | `"project"` included |
| Parent dirs | `CLAUDE.md` in directories above `cwd` | `"project"`, at session start |
| Child dirs | `CLAUDE.md` in subdirectories of `cwd` | `"project"`, **on demand** |
| Local | `CLAUDE.local.md` | `"local"` included |
| User | `~/.claude/CLAUDE.md`, `~/.claude/rules/*.md` | `"user"` included |

All levels are **additive**. If they conflict there is no hard precedence rule, so write non-conflicting rules or state precedence explicitly ("These project instructions override user-level defaults").

> You can also inject context straight from code with `system_prompt`, with no files needed. Use `CLAUDE.md` when you want the **same** instructions shared between interactive Claude Code sessions and your SDK agents.

## 6. What `setting_sources` does **not** control

A few inputs are read **regardless** of `setting_sources`:

| Input | Behavior | How to turn it off |
|---|---|---|
| Managed policy settings (MDM, registry, managed file, server-managed) | Always applied | Controlled by the machine or org, not by the SDK |
| `~/.claude.json` global config | Always read | Relocate with `CLAUDE_CONFIG_DIR` in `env` |
| Auto memory at `~/.claude/projects/<project>/memory/` | Loaded into the system prompt | `CLAUDE_CODE_DISABLE_AUTO_MEMORY=1` in `env` (or `autoMemoryEnabled: false`) |
| claude.ai MCP connectors | Loaded when you authenticate with a claude.ai login | `ENABLE_CLAUDEAI_MCP_SERVERS=false` in `env`, or `disableClaudeAiConnectors` in settings |
| `sandbox.credentials` deny/mask entries in user settings | Kept as restrictions when the command sandbox runs | Remove them from `~/.claude/settings.json` |

> **Warning for multi-tenant apps:** do **not** rely on default `query()` options for isolation. Run each tenant in its own filesystem and set `setting_sources=[]` plus `CLAUDE_CODE_DISABLE_AUTO_MEMORY=1` in `env`.

Here is what a locked-down configuration looks like:

In [ ]:
isolated_options = ClaudeAgentOptions(
    setting_sources=[],                                   # no user/project/local files
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},         # no per-directory memory
    cwd=str(DEMO_DIR),
)
print(isolated_options.setting_sources, isolated_options.env)

## 7. Skills in more depth

- Discovered through `setting_sources` from `.claude/skills/<name>/SKILL.md`.
- If you **omit** the `skills` option, discovered user and project skills are enabled and the `Skill` tool is available (like the CLI).
- `skills="all"`, a **list of names**, or `[]` (disable all) lets you control which are active.
- If you also pass an explicit `tools=[...]` list, include `"Skill"` in it.

A skill description is the *trigger*. Write it like a sentence that says **when** to use the skill, as we did ("Use when the user asks what a flurbo is...").

Select just one skill by name:

In [ ]:
answer_d, tools_d = await ask(
    QUESTION,
    setting_sources=["project"],
    skills=["flurbo-glossary"],
    allowed_tools=["Read", "Glob"],
)

## 8. Hooks: two ways to define them

| Hook type | Defined in | Best for |
|---|---|---|
| **Filesystem** | `settings.json` (loaded via `setting_sources`) | Sharing hooks between the CLI and SDK. Types: `command` (shell), `http`, `mcp_tool`, `prompt` (LLM check), `agent` (verifier agent) |
| **Programmatic** | Callbacks in `ClaudeAgentOptions(hooks=...)` | App-specific logic, structured decisions, in-process integration |

Both fire in the main agent **and in subagents**. Programmatic callbacks receive `agent_id` and `agent_type` so you can tell which agent fired.

A hook callback gets `(input_data, tool_use_id, context)` and returns:
- `{}` to **allow** the tool call, or
- `{"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "deny", "permissionDecisionReason": "..."}}` to **block**. Claude receives the reason as the tool result.

### 8a. Test a hook *without* the API
A hook is just an async function, so you can unit-test it directly with fake input. No tokens, no agent.

In [ ]:
# OFFLINE
import asyncio

async def audit_bash(input_data, tool_use_id, context):
    command = input_data.get("tool_input", {}).get("command", "")
    if "rm -rf" in command:
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": "Destructive command blocked",
            }
        }
    return {}   # empty dict: allow

cases = [
    {"tool_name": "Bash", "tool_input": {"command": "ls -la"}},
    {"tool_name": "Bash", "tool_input": {"command": "rm -rf /tmp/anything"}},
    {"tool_name": "Bash", "tool_input": {}},
]
for case in cases:
    decision = await audit_bash(case, "toolu_fake", None)
    verdict = decision.get("hookSpecificOutput", {}).get("permissionDecision", "allow")
    print(f"{case['tool_input'].get('command', '(no command)'):28} -> {verdict}")

### 8b. Programmatic hook, live
Register the hook with a `HookMatcher`. `matcher="Bash"` limits it to the `Bash` tool. The command we ask for only *prints* the dangerous words, so nothing can be harmed even if the hook were bypassed.

In [ ]:
answer_e, _ = await ask(
    "Run this exact shell command and show its output: echo 'rm -rf demo'",
    allowed_tools=["Bash"],
    permission_mode="acceptEdits",
    hooks={"PreToolUse": [HookMatcher(matcher="Bash", hooks=[audit_bash])]},
)

Expect the agent to say the command was **blocked** ("Destructive command blocked"). The hook ran in *your* Python process, outside the context window.

### 8c. Filesystem hook (`settings.json`)
The same idea, but defined as a **shell command** in `.claude/settings.json`. It appends a line to a log file every time `Bash` is about to run. The SDK picks it up because we pass `setting_sources=["project"]`.

In [ ]:
import json

HOOK_LOG = DEMO_DIR / "bash_hook.log"
HOOK_LOG.unlink(missing_ok=True)

settings = {
    "hooks": {
        "PreToolUse": [
            {
                "matcher": "Bash",
                "hooks": [
                    {"type": "command", "command": f'echo "bash about to run" >> "{HOOK_LOG}"'}
                ],
            }
        ]
    }
}
(DEMO_DIR / ".claude").mkdir(exist_ok=True)
(DEMO_DIR / ".claude" / "settings.json").write_text(json.dumps(settings, indent=2))
print((DEMO_DIR / ".claude" / "settings.json").read_text())

In [ ]:
await ask(
    "Run `echo hello` in the shell and show the output.",
    setting_sources=["project"],
    skills=[],
    allowed_tools=["Bash"],
    permission_mode="acceptEdits",
)

print("\nhook log exists:", HOOK_LOG.exists())
if HOOK_LOG.exists():
    print(HOOK_LOG.read_text())

If the log is empty, check that `setting_sources` includes `"project"` and that `cwd` is the folder that contains `.claude/`. Project hooks load only from `<cwd>/.claude/`.

The TypeScript SDK supports extra hook events that Python does not yet (`SessionStart`, `SessionEnd`, `TeammateIdle`, `TaskCompleted`).

## 9. Choose the right feature

| What you want | Use | SDK surface |
|---|---|---|
| Conventions the agent **always** follows | `CLAUDE.md` | `setting_sources=["project"]` |
| Reference material loaded **when relevant** | Skills | `setting_sources` + `skills` |
| A reusable workflow (deploy, review, release) | User-invocable skills | `setting_sources` + `skills` |
| Hand an isolated subtask to a **fresh context** | Subagents | `agents` + `"Agent"` in `allowed_tools` |
| Several Claude Code instances sharing tasks and messages | Agent teams | A CLI feature (not configured through SDK options) |
| Deterministic logic on tool calls (audit, block, transform) | Hooks | `hooks=` callbacks, or shell hooks via `setting_sources` |
| Structured access to an external service | MCP | `mcp_servers=` |
| Context from code only, no files | `system_prompt` | `system_prompt=` |

> Every feature you enable adds to the context window. Turn on only what the agent needs.

## 10. Recap

- `setting_sources` decides which `user` / `project` / `local` files the SDK loads. Omitted means **all three**; `[]` means **none**.
- **`CLAUDE.md`** is persistent instruction text, loaded in full every session.
- **Skills** are loaded **on demand**; only their descriptions load at startup.
- **Hooks** can be shell commands in `settings.json` or Python callbacks; both can allow or block tool calls.
- Some inputs (managed policy, auto memory, claude.ai connectors) load **regardless**, so isolate tenants deliberately.

### Try it yourself
1. Add a second skill (for example `release-checklist`) and ask something that should trigger it. Does the agent pick the right one?
2. Create `.claude/rules/style.md` with a formatting rule and confirm it applies.
3. Add `"user"` to `setting_sources` and notice what changes once your own `~/.claude` files load.
4. Write a hook that denies `Write` to any path outside the demo folder.
5. Next: **`agent-sdk-sessions-continue-resume-fork.ipynb`**.